<div style="padding: 20px; background: linear-gradient(90deg, #b92b27 0%, #1565C0 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">👨‍👦 Module 6.2: Parent Document Retriever</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Solving the Chunking Paradox with hierarchical retrieval.</p>
</div>

---

## 1. The Chunking Paradox

In RAG, chunk size is a major dilemma:
- **Small Chunks (e.g., 1 sentence)** are mathematically brilliant for Vector Search. They match queries perfectly. But when you pass them to the LLM, they lack surrounding context, so the LLM gives bad answers.
- **Large Chunks (e.g., 3 paragraphs)** give the LLM great context, but they are terrible for Vector Search. The mathematical meaning of the chunk gets "diluted" by too many words, making it hard to find.

## 2. The Solution: Parent-Child Hierarchy
The **Parent Document Retriever** does both:
1. It splits your document into **Tiny Child Chunks** and embeds those in the Vector DB.
2. It links those children to the **Massive Parent Document** using a `parent_id`.
3. When a user searches, we do the math on the tiny children (high accuracy), but we return the Massive Parent to the LLM (high context)!

Let's build this mechanism from scratch to see exactly how it works.

In [1]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import uuid
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 1. Create a massive Parent Document
parent_text = (
    "LangChain is an open-source framework built for LLM applications. "
    "It provides tools for routing, retrieval, and agents. "
    "The framework was launched by Harrison Chase in late 2022. "
    "Vector Stores like Chroma and FAISS integrate seamlessly with it. "
    "You can use HuggingFace models for free local embeddings. "
    "Groq provides lightning fast inference for generation."
)

parent_id = str(uuid.uuid4())

# 2. Save the massive parent safely in a standard dictionary (No need to embed it!)
docstore = {
    parent_id: parent_text
}

# 3. Split the text into tiny child fragments
child_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
child_chunks = child_splitter.split_text(parent_text)

# 4. Embed only the children into Chroma, tagging them with the parent_id
child_docs = [
    Document(page_content=chunk, metadata={"parent_id": parent_id}) 
    for chunk in child_chunks
]

vectorstore = Chroma.from_documents(
    child_docs, 
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    collection_name="parent_child_demo"
)
print(f"Ingested {len(child_docs)} tiny child chunks pointing to 1 massive parent.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ingested 8 tiny child chunks pointing to 1 massive parent.


## 3. Proving the Concept
Watch what happens when we search! We find the tiny child, grab its `parent_id`, and pull the massive parent for the LLM.

In [2]:
query = "Who created the framework?"

# STEP 1: Search the Vector DB (What the math sees)
raw_vector_results = vectorstore.similarity_search(query, k=1)
best_child = raw_vector_results[0]

print("--- STEP 1: VECTOR DB RESULT (The Tiny Child) ---")
print(f"Text: {best_child.page_content}")
print(f"Metadata: {best_child.metadata}\n")

# STEP 2: Lookup the Parent (What the LLM sees)
retrieved_parent_id = best_child.metadata["parent_id"]
parent_context = docstore[retrieved_parent_id]

print("--- STEP 2: FINAL RETRIEVED CONTEXT (The Massive Parent) ---")
print(parent_context)

--- STEP 1: VECTOR DB RESULT (The Tiny Child) ---
Text: retrieval, and agents. The framework was launched
Metadata: {'parent_id': '5fb5e1bb-4000-47f0-b91c-bcdb51e15650'}

--- STEP 2: FINAL RETRIEVED CONTEXT (The Massive Parent) ---
LangChain is an open-source framework built for LLM applications. It provides tools for routing, retrieval, and agents. The framework was launched by Harrison Chase in late 2022. Vector Stores like Chroma and FAISS integrate seamlessly with it. You can use HuggingFace models for free local embeddings. Groq provides lightning fast inference for generation.
